In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyxdf
import os
from pathlib import Path

In [52]:
xdf_path = Path(r"C:\Users\jshin\OW_closedloopLIFU\xdf_data\sub-dave_run_4\ses-1\eeg\sub-dave_run_4_ses-1_task-dave_run_4_run-001_eeg.xdf")
data, header = pyxdf.load_xdf(str(xdf_path))

In [53]:
stream = data[2]
df= []
df = pd.DataFrame(stream['time_series'])
df = df.rename(columns={i: f"Ch{i}" for i in range(df.shape[1])})

# Add timestamp column
df['Timestamp'] = stream['time_stamps']

# Move Timestamp to be the first column after the index
cols = ['Timestamp'] + [col for col in df.columns if col != 'Timestamp']
df = df[cols]
eeg_raw = df#[['Ch0', 'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch12','Timestamp']]
eeg_raw

In [54]:
EEG_LIFU_events = data[0]

# Create DataFrame from time_series
markers = pd.DataFrame(EEG_LIFU_events['time_series'])

# Rename the first column to 'markers'
markers.rename(columns={0: 'markers'}, inplace=True)

# Add timestamp column
markers['Timestamp'] = EEG_LIFU_events['time_stamps']

# Move Timestamp to be the first column after the index
cols = ['Timestamp'] + [col for col in markers.columns if col != 'Timestamp']
markers = markers[cols]
markers

In [55]:
lifu_on = markers[markers['markers'] == 'LIFU_ON']
lifu_on = np.array(lifu_on['Timestamp'])
lifu_on

In [56]:
pre = 0
post = 2
fs = 250  # sampling rate in Hz -- adjust if this isn't right for your data
pre_samples = int(pre * fs)    # 500
post_samples = int(post * fs)  # 1750
window_len = pre_samples + post_samples  # 2250

raw = eeg_raw.reset_index(drop=True)
timestamps = raw["Timestamp"].values

raw_windows = []
for idx, event in enumerate(lifu_on):
    center_idx = np.abs(timestamps - event).argmin()  # closest sample to onset
    start_idx = center_idx - pre_samples
    end_idx = start_idx + window_len

    if start_idx < 0 or end_idx > len(raw):
        print(f"Skipping event {idx}: window out of bounds ({start_idx}:{end_idx})")
        continue

    window_df = raw.iloc[start_idx:end_idx].copy()
    window_df["t_rel"] = np.arange(-pre_samples, post_samples) / fs
    window_df["idx"] = idx
    raw_windows.append(window_df)

raw_windows[0]

In [57]:
pre = 0
post = 2
fs = 250
window_len = int((pre + post) * fs)
index = 0

# Take the first window (one event)
window_df = raw_windows[index]

t = window_df["t_rel"].values   # relative time axis

plt.figure(figsize=(10,6))

# Loop over EEG channels in this window
for i, ch in enumerate([c for c in window_df.columns if c not in ["Timestamp","t_rel","idx"]]):
    signal = window_df[ch].values
    signal_scaled = (signal - signal.mean()) / signal.std()
    plt.plot(t, signal_scaled + i, label=ch)

plt.xlabel("Time (s)")
plt.ylabel("Channel")
plt.title(f"EEG channels in sonication {index+1} (scaled, offset)")
plt.yticks(range(len(window_df.columns)-3), [c for c in window_df.columns if c not in ["Timestamp","t_rel","idx"]])
plt.legend(loc="upper right")
plt.show()
